# 11 Differential Abundance

In notebooks 08–10, we tested whether fungal community composition was associated with settlement type, diet, or age. Here, we test whether fungal genus abundances are associated with settlement type or age, using the unrarefied ITS1 ASV counts and matching taxonomy from notebook 03.

In [3]:
import pandas as pd
from qiime2 import Artifact, Visualization

In [4]:
in_dir = "../data/processed_ignored/03_taxonomy"
out_dir = "../data/processed_ignored/11_differential_abundance"
vis_dir = "../data/processed/11_differential_abundance"

fungal_table = f"{in_dir}/table_fungi_filtered.qza"
taxonomy = f"{in_dir}/taxonomy.qza"

genus_table = f"{out_dir}/table_fungi_l6.qza"
named_genus_ids = f"{out_dir}/named_genus_ids.tsv"

metadata = "../data/processed_ignored/06_diversity/Final_metadata_with_GHSL.tsv"

! mkdir -p "$out_dir" "$vis_dir"

In [5]:
# 01 - DA by settlement

settlement_out_dir = f"{out_dir}/settlement"
settlement_vis_dir = f"{vis_dir}/settlement"

settlement_table = f"{settlement_out_dir}/table_fungi_l6_settlement.qza"
named_genus_table = f"{settlement_out_dir}/table_fungi_l6_settlement_named.qza"

# ANCOM-BC rural
settlement_differentials = f"{settlement_out_dir}/ancombc_settlement_l6_differentials.qza"
settlement_barplot = f"{settlement_vis_dir}/ancombc_settlement_l6_barplot.qzv"
settlement_results = f"{settlement_vis_dir}/ancombc_settlement_l6_results.qzv"

# ANCOM-BC urban
urban_ref_differentials = f"{settlement_out_dir}/ancombc_settlement_l6_urban_ref.qza"
urban_ref_barplot = f"{settlement_vis_dir}/ancombc_settlement_l6_urban_ref_barplot.qzv"
urban_ref_results = f"{settlement_vis_dir}/ancombc_settlement_l6_urban_ref_results.qzv"

! mkdir -p "$settlement_out_dir" "$settlement_vis_dir"

In [19]:
# DA by age

age_out_dir = f"{out_dir}/age"
age_vis_dir = f"{vis_dir}/age"

age_metadata = f"{age_out_dir}/age_metadata.tsv"
age_sample_table = f"{age_out_dir}/table_fungi_l6_age.qza"
age_named_genus_table = f"{age_out_dir}/table_fungi_l6_age_named.qza"

# ANCOM-BC
age_differentials = f"{age_out_dir}/ancombc2_age_l6_differentials.qza"
age_barplot = f"{age_vis_dir}/ancombc2_age_l6_barplot.qzv"
age_results = f"{age_vis_dir}/ancombc2_age_l6_results.qzv"

! mkdir -p "$age_out_dir" "$age_vis_dir"

In [21]:
# DA by diet

diet_out_dir = f"{out_dir}/diet"
diet_vis_dir = f"{vis_dir}/diet"

diet_sample_table = f"{diet_out_dir}/table_fungi_l6_diet.qza"
diet_named_genus_table = f"{diet_out_dir}/table_fungi_l6_diet_named.qza"
diet_differentials = f"{diet_out_dir}/ancombc_diet_l6_differentials.qza"
diet_barplot = f"{diet_vis_dir}/ancombc_diet_l6_barplot.qzv"
diet_results = f"{diet_vis_dir}/ancombc_diet_l6_results.qzv"

! mkdir -p "$diet_out_dir" "$diet_vis_dir"

## 01 - Differential abundance by settlement

### Check that fungal ASVs have taxonomy entries

We compare the feature IDs in the unrarefied fungal count table with those in the taxonomy. The taxonomy can contain extra IDs because the count table was filtered to retain fungi.

In [55]:
counts = Artifact.load(fungal_table).view(pd.DataFrame)
tax = Artifact.load(taxonomy).view(pd.DataFrame)

missing_ids = counts.columns.difference(tax.index)

print("ASVs in the fungal count table:", len(counts.columns))
print("ASVs without a taxonomy entry:", len(missing_ids))

ASVs in the fungal count table: 575
ASVs without a taxonomy entry: 0


### Match the count table to the metadata

We compare sample IDs in the unrarefied count table and the cleaned metadata from notebook 06 to see how many samples are available for metadata-based analyses.

In [5]:
meta = pd.read_csv(metadata, sep="\t", index_col=0, dtype=str)
matched_ids = counts.index.intersection(meta.index)

print("Samples in the unrarefied count table:", len(counts.index))
print("Samples with a metadata entry:", len(matched_ids))

Samples in the unrarefied count table: 150
Samples with a metadata entry: 150


### Summarize metadata for the matched samples

We count samples in each settlement and diet group, including missing values. We also check how many samples have a numeric age.

In [6]:
matched_meta = meta.loc[matched_ids]

print("Settlement type")
print(matched_meta["Settlement_Type"].value_counts(dropna=False))

print("\nDiet")
print(matched_meta["diet_type_sample"].value_counts(dropna=False))

age = pd.to_numeric(matched_meta["age_years_sample"], errors="coerce")
print("\nSamples with a numeric age:", age.count())

Settlement type
Settlement_Type
Rural       65
Urban       56
Suburban    24
NaN          5
Name: count, dtype: int64

Diet
diet_type_sample
Omnivore                            118
Vegetarian but eat seafood            8
Omnivore but do not eat red meat      8
Vegetarian                            6
Vegan                                 5
NaN                                   5
Name: count, dtype: int64

Samples with a numeric age: 145


### Select samples with settlement information

We select samples labeled Rural, Suburban, or Urban for the settlement analysis.

In [7]:
settlement_groups = ["Rural", "Suburban", "Urban"]
settlement_meta = matched_meta.loc[
    matched_meta["Settlement_Type"].isin(settlement_groups)
].copy()

print("Samples in the settlement analysis:", len(settlement_meta))

Samples in the settlement analysis: 145


### Inspect fungal taxonomy assignments

We look at the taxonomy labels for the ASVs detected in the settlement samples to see which ranks are available.

In [8]:
settlement_counts = counts.loc[settlement_meta.index]
detected_asvs = settlement_counts.columns[settlement_counts.gt(0).any(axis=0)]

with pd.option_context("display.max_colwidth", None):
    display(tax.loc[detected_asvs, ["Taxon"]].head(8))

,Taxon
f3965ee56e352f1f41274f3b2b4e9a7c,k__Fungi;p__Ascomycota;c__Eurotiomycetes;o__Eurotiales;f__Aspergillaceae;g__Penicillium;s__Penicillium_nodulum
726d70b2590173b188b39dda229fc0bb,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales
48c9b5109cb8dd3c673c7936bd72e7ea,k__Fungi;p__Mucoromycota;c__Mucoromycetes;o__Mucorales;f__Mucoraceae;g__Mucor;s__Mucor_racemosus
6ccbc16d6454d876ebb3cc5428d78b65,k__Fungi;p__Ascomycota;c__Eurotiomycetes;o__Chaetothyriales;f__Herpotrichiellaceae;g__Exophiala;s__Exophiala_sp
ab3002d65dd4e784bab7aa764abead9b,k__Fungi;p__Ascomycota;c__Dothideomycetes;o__Pleosporales;f__Pleosporaceae;g__Asteromyces;s__Asteromyces_cruciatus
d4f7289b18cf32a14ffda597b4aa422b,k__Fungi;p__Basidiomycota;c__Cystobasidiomycetes;o__Cystobasidiales;f__Cystobasidiaceae;g__Cystobasidium;s__Cystobasidium_slooffiae
7993a769e465179581bb7a6a5e3adbe9,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetales_fam_Incertae_sedis;g__Candida;s__Candida_albicans
e02d076ae4ab5f6fba989ef48581f288,k__Fungi;p__Ascomycota;c__Eurotiomycetes;o__Chaetothyriales;f__Trichomeriaceae;g__Strelitziana;s__Strelitziana_syzygii


### Check genus-level coverage

We count how many detected ASVs have a genus label and what fraction of fungal reads those ASVs represent across the settlement samples.

In [9]:
labels = tax.loc[detected_asvs, "Taxon"]
genus = labels.str.split(";g__", regex=False).str[1].str.split(";").str[0]
has_genus = genus.fillna("").str.strip().ne("")

reads_per_asv = settlement_counts.loc[:, detected_asvs].sum(axis=0)
genus_read_fraction = reads_per_asv[has_genus].sum() / reads_per_asv.sum()

print(f"ASVs with a genus label: {has_genus.sum()} / {len(detected_asvs)}")
print(f"Reads from ASVs with a genus label: {genus_read_fraction:.1%}")

ASVs with a genus label: 507 / 562
Reads from ASVs with a genus label: 97.9%


### Collapse the feature table to genus level

We use `qiime taxa collapse` to sum the unrarefied ASV counts at taxonomic level 6. The resulting table contains all samples from notebook 03. We will select the settlement samples in a separate step.

In [54]:
! qiime taxa collapse \
    --i-table "$fungal_table" \
    --i-taxonomy "$taxonomy" \
    --p-level 6 \
    --o-collapsed-table "$genus_table"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/11_differential_abundance/table_fungi_l6.qza


### Check genus labels in the collapsed table

We check whether the collapsed table contains any features without a named genus. We look at the last rank in each feature label because the collapse was set to genus level.

In [42]:
collapsed_counts = Artifact.load(genus_table).view(pd.DataFrame)

labels = collapsed_counts.columns.to_series()
last_rank = labels.str.split(";").str[-1].str.strip()
named_genus = last_rank.str.startswith("g__") & last_rank.ne("g__")

print("Features in the collapsed table:", len(labels))
print("Features without a named genus:", (~named_genus).sum())
print(labels[~named_genus].head(5).to_string(index=False))

Features in the collapsed table: 184
Features without a named genus: 29
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Sa...
k__Fungi;p__Mortierellomycota;c__Mortierellomyc...
k__Fungi;p__Basidiomycota;c__Agaricomycetes;__;...
k__Fungi;p__Basidiomycota;c__Agaricomycetes;o__...
k__Fungi;p__Ascomycota;c__Dothideomycetes;o__Pl...


### Filter the genus-level table by settlement type

We retain samples labeled Rural, Suburban, or Urban in the cleaned metadata.

In [43]:
! qiime feature-table filter-samples \
    --i-table "$genus_table" \
    --m-metadata-file "$metadata" \
    --p-where "Settlement_Type IN ('Rural', 'Suburban', 'Urban')" \
    --o-filtered-table "$settlement_table"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/11_differential_abundance/settlement/table_fungi_l6_settlement.qza


### Keep features with a named genus

We save the feature IDs that end in a named genus, then use that list to filter the settlement table.

In [44]:
last_rank[named_genus].rename("Genus").to_frame().to_csv(
    named_genus_ids, sep="\t", index_label="feature-id"
)

! qiime feature-table filter-features \
    --i-table "$settlement_table" \
    --m-metadata-file "$named_genus_ids" \
    --o-filtered-table "$named_genus_table"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/11_differential_abundance/settlement/table_fungi_l6_settlement_named.qza


### Run ANCOM-BC for settlement type

We test whether fungal genus abundances differ by settlement type. Rural is the reference group, so the model estimates two comparisons: Suburban versus Rural and Urban versus Rural.

In [45]:
! qiime composition ancombc \
    --i-table "$named_genus_table" \
    --m-metadata-file "$metadata" \
    --p-formula Settlement_Type \
    --p-reference-levels "Settlement_Type::Rural" \
    --o-differentials "$settlement_differentials"

Saved FeatureData[DifferentialAbundance] to: ../data/processed_ignored/11_differential_abundance/settlement/ancombc_settlement_l6_differentials.qza


### Generate the differential abundance barplot

We plot the estimated log-fold changes for Suburban versus Rural and Urban versus Rural. Positive values indicate higher estimated abundance in the comparison group, and negative values indicate lower estimated abundance.

In [46]:
! qiime composition da-barplot \
    --i-data "$settlement_differentials" \
    --o-visualization "$settlement_barplot"

Saved Visualization to: ../data/processed/11_differential_abundance/settlement/ancombc_settlement_l6_barplot.qzv


### Generate the differential abundance results table

We tabulate the ANCOM-BC results to inspect the estimated log-fold changes and adjusted p-values (q-values) for each genus and comparison.

In [47]:
! qiime composition tabulate \
    --i-data "$settlement_differentials" \
    --o-visualization "$settlement_results"

Saved Visualization to: ../data/processed/11_differential_abundance/settlement/ancombc_settlement_l6_results.qzv


In [ ]:
Visualization.load(settlement_barplot)

### Settlement type results

ANCOM-BC identified one genus, *Oligophagozyma*, with higher estimated abundance in Suburban than in Rural samples (log-fold change = +1.152, q = 0.04686). No genus passed the q < 0.05 cutoff for Urban versus Rural. Rural was the reference group for both comparisons.

### Visualize the Suburban-versus-Urban results

We rerun the settlement model to generate the ANCOM-BC barplot and results table with Urban as the reference group. In the barplot, select `Settlement_TypeSuburban` and apply the `q < 0.05` filter. Positive log-fold changes indicate higher estimated abundance in Suburban samples, and negative values indicate higher estimated abundance in Urban samples.

In [48]:
! qiime composition ancombc \
    --i-table "$named_genus_table" \
    --m-metadata-file "$metadata" \
    --p-formula Settlement_Type \
    --p-reference-levels "Settlement_Type::Urban" \
    --o-differentials "$urban_ref_differentials"

! qiime composition da-barplot \
    --i-data "$urban_ref_differentials" \
    --o-visualization "$urban_ref_barplot"

! qiime composition tabulate \
    --i-data "$urban_ref_differentials" \
    --o-visualization "$urban_ref_results"

Saved FeatureData[DifferentialAbundance] to: ../data/processed_ignored/11_differential_abundance/settlement/ancombc_settlement_l6_urban_ref.qza
Saved Visualization to: ../data/processed/11_differential_abundance/settlement/ancombc_settlement_l6_urban_ref_barplot.qzv
Saved Visualization to: ../data/processed/11_differential_abundance/settlement/ancombc_settlement_l6_urban_ref_results.qzv


In [ ]:
Visualization.load(urban_ref_barplot)

### Suburban versus Urban results

With Urban as the reference group, no fungal genus passed the adjusted significance cutoff (q < 0.05) for Suburban versus Urban in the genus-level ANCOM-BC analysis.

### Settlement type discussion

The whole-ASV beta diversity analysis detected differences in community composition for Rural versus Suburban and Suburban versus Urban. In the genus-level ANCOM-BC analysis, only *Oligophagozyma* passed the adjusted significance cutoff for Suburban versus Rural, and no genus passed it for Suburban versus Urban. The two analyses address different questions: beta diversity tests differences across the community as a whole, while ANCOM-BC tests each genus individually.

## 02 - Differential abundance by age

### Select samples with numeric age and retain named genera

We start from the full unrarefied genus-level table and select samples with a numeric `age_years_sample`. We keep age in years as a continuous variable and reuse the same named-genus filter as in the settlement analysis.

In [13]:
age_meta = pd.read_csv(metadata, sep="\t", index_col=0, dtype=str)
age = pd.to_numeric(age_meta["age_years_sample"], errors="coerce")

age.dropna().to_frame().to_csv(
    age_metadata, sep="\t", index_label="sample-id"
)

! qiime feature-table filter-samples \
    --i-table "$genus_table" \
    --m-metadata-file "$age_metadata" \
    --o-filtered-table "$age_sample_table"

! qiime feature-table filter-features \
    --i-table "$age_sample_table" \
    --m-metadata-file "$named_genus_ids" \
    --o-filtered-table "$age_named_genus_table"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/11_differential_abundance/age/table_fungi_l6_age.qza
Saved FeatureTable[Frequency] to: ../data/processed_ignored/11_differential_abundance/age/table_fungi_l6_age_named.qza


### Run ANCOM-BC2 for age

We use ANCOM-BC2 to test whether fungal genus abundance is associated with age. The model treats age in years as a continuous predictor and estimates the change in log abundance per additional year of age for each genus.

In [20]:
! qiime composition ancombc2 \
    --i-table "$age_named_genus_table" \
    --m-metadata-file "$age_metadata" \
    --p-fixed-effects-formula age_years_sample \
    --o-ancombc2-output "$age_differentials"

! qiime composition da-barplot \
    --i-data "$age_differentials" \
    --o-visualization "$age_barplot"

! qiime composition tabulate \
    --i-data "$age_differentials" \
    --o-visualization "$age_results"

Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/11_differential_abundance/age/ancombc2_age_l6_differentials.qza
Saved Visualization to: ../data/processed/11_differential_abundance/age/ancombc2_age_l6_barplot.qzv
Saved Visualization to: ../data/processed/11_differential_abundance/age/ancombc2_age_l6_results.qzv


In [ ]:
Visualization.load(age_barplot)

### Age results

Two fungal genera passed the adjusted significance cutoff (`q < 0.05`) in the genus-level ANCOM-BC2 analysis: 
- *Kluyveromyces* had higher estimated abundance in samples from older participants (age coefficient ≈ +0.062 log-abundance units per year, q = 0.000545). 
- *Clavispora* had lower estimated abundance in samples from older participants (age coefficient ≈ −0.058 log-abundance units per year, q = 0.000480).

### Age discussion

In notebook 10, age explained a small share of the variation in overall fungal community composition (1.5% for 16-mer Bray–Curtis and 1.1% for whole-ASV Bray–Curtis). The genus-level analysis identified two specific associations with age. Estimated *Kluyveromyces* abundance was higher in samples from older participants, while estimated *Clavispora* abundance was lower. The model included age as its only predictor, so these associations are unadjusted for diet and settlement type.

## 03 - Differential abundance by diet

### Filter by diet and run ANCOM-BC

We retain the five original diet groups and compare each of the four smaller groups with Omnivore.

In [23]:
! qiime feature-table filter-samples \
    --i-table "$genus_table" \
    --m-metadata-file "$metadata" \
    --p-where "diet_type_sample IN ('Omnivore', 'Omnivore but do not eat red meat', 'Vegetarian but eat seafood', 'Vegetarian', 'Vegan')" \
    --o-filtered-table "$diet_sample_table"

! qiime feature-table filter-features \
    --i-table "$diet_sample_table" \
    --m-metadata-file "$named_genus_ids" \
    --o-filtered-table "$diet_named_genus_table"

! qiime composition ancombc \
    --i-table "$diet_named_genus_table" \
    --m-metadata-file "$metadata" \
    --p-formula diet_type_sample \
    --p-reference-levels "diet_type_sample::Omnivore" \
    --o-differentials "$diet_differentials"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/11_differential_abundance/diet/table_fungi_l6_diet.qza
Saved FeatureTable[Frequency] to: ../data/processed_ignored/11_differential_abundance/diet/table_fungi_l6_diet_named.qza
Saved FeatureData[DifferentialAbundance] to: ../data/processed_ignored/11_differential_abundance/diet/ancombc_diet_l6_differentials.qza


### Visualize the diet results

In [24]:
! qiime composition da-barplot \
    --i-data "$diet_differentials" \
    --o-visualization "$diet_barplot"

! qiime composition tabulate \
    --i-data "$diet_differentials" \
    --o-visualization "$diet_results"

Saved Visualization to: ../data/processed/11_differential_abundance/diet/ancombc_diet_l6_barplot.qzv
Saved Visualization to: ../data/processed/11_differential_abundance/diet/ancombc_diet_l6_results.qzv


In [ ]:
Visualization.load(diet_barplot)

### Diet results with Omnivore as the reference

In the exploratory genus-level ANCOM-BC analysis, six fungal genera passed the adjusted significance cutoff (`q < 0.05`) in at least one of the four comparisons with the Omnivore reference group. *Pleurotus* had higher estimated abundance in the “Omnivore but do not eat red meat” group and lower estimated abundance in the Vegan group. *Apiotrichum* had higher estimated abundance in the Vegetarian group. The three genera that passed the cutoff in the “Vegetarian but eat seafood” group all had lower estimated abundance. However, the small, unequal diet groups limit how broadly these findings can be generalized.

| Diet group compared with Omnivore | Genus | Log-fold change | Adjusted q-value |
|---|---|---:|---:|
| Omnivore but do not eat red meat | *Pleurotus* | +0.792 | 0.01073 |
| Vegetarian | *Apiotrichum* | +0.528 | 0.04643 |
| Vegetarian but eat seafood | *Debaryomyces* | −0.599 | 0.003715 |
| Vegetarian but eat seafood | *Oligophagozyma* | −0.684 | 0.002280 |
| Vegetarian but eat seafood | *Mucor* | −0.781 | 0.0005837 |
| Vegan | *Pleurotus* | −1.667 | 2 × 10⁻¹⁰ |
| Vegan | *Oligophagozyma* | −2.077 | Displayed as 0¹ |
| Vegan | *Penicillium* | −2.586 | 0.0002495 |

¹ The QIIME 2 barplot displays this q-value as `0`. The exact value could not be read from the plot.

## Conclusion

At the genus level, the differential abundance analyses identified one association with settlement type and two with age. In the diet analysis, six genera differed in estimated abundance in at least one comparison with the Omnivore reference group. These diet findings should be interpreted cautiously given the small and unequal group sizes.

The diet results illustrate why beta diversity and differential abundance answer different questions. The Bray–Curtis PERMANOVA did not detect a statistically significant difference in overall fungal community composition across diet groups. That community-level test does not estimate how the abundance of any individual genus differs between groups. Differential abundance analysis tests those genus-level differences directly, so individual genera can show associations even when the overall community test is nonsignificant.